# 🧑‍⚖️ Avatar Realista — Neto de Moraes Associados
Fluxo SIMPLES (que já funcionou): **foto** + **áudio** → **vídeo do avatar falando**.

1. Runtime → Change runtime type → **T4 GPU** → Save
2. **Ambiente de execução → Executar tudo**
3. Na célula 2, envie **as fotos** e depois **o áudio** (10–60s da sua voz)
4. A última célula baixa o vídeo


In [ ]:
# 1) Prepara o SadTalker (ambiente padrão do Colab)
!rm -rf /content/SadTalker
!git clone -q https://github.com/OpenTalker/SadTalker /content/SadTalker
%cd /content/SadTalker
# --- patches de compatibilidade (numpy 2.x do Colab) ---
!sed -i 's/np.VisibleDeprecationWarning/Warning/g' src/face3d/util/preprocess.py
!grep -rl "np\.float\b" src | xargs -r sed -i 's/np\.float\b/float/g'
!grep -rl "np\.int\b" src | xargs -r sed -i 's/np\.int\b/int/g'
!grep -rl "np\.bool\b" src | xargs -r sed -i 's/np\.bool\b/bool/g'
!grep -rl "np\.float\b" /usr/local/lib/python3*/dist-packages/basicsr /usr/local/lib/python3*/dist-packages/facexlib /usr/local/lib/python3*/dist-packages/gfpgan 2>/dev/null | xargs -r sed -i 's/np\.float\b/float/g'
!grep -rl "functional_tensor import" /usr/local/lib/python3*/dist-packages/basicsr/data/degradations.py 2>/dev/null | xargs -r sed -i 's/functional_tensor import/functional import/g'
!bash scripts/download_models.sh 2>&1 | tail -2
import numpy
print("numpy:", numpy.__version__, "| OK: SadTalker pronto")
print("OK: SadTalker pronto")

In [ ]:
# 2) Envie as FOTOS (pode ser mais de uma) e depois o AUDIO
from google.colab import files
import os, shutil
os.makedirs('/content/input', exist_ok=True)
!rm -f /content/input/*

print("PASSO 1 - Envie as FOTOS (pode selecionar varias):")
up1 = files.upload()
photos = []
for f in up1:
    dst = '/content/input/' + f
    shutil.move(f, dst)
    if os.path.splitext(f)[1].lower() in ('.jpg','.jpeg','.png'):
        photos.append(dst)
if not photos:
    raise ValueError("Nenhuma foto enviada. Rode esta celula de novo.")
photo = max(photos, key=os.path.getsize)
print(f"{len(photos)} foto(s). Principal: {photo}")

print("\nPASSO 2 - Envie o AUDIO da sua voz (10-60s):")
up2 = files.upload()
audio = None
for f in up2:
    dst = '/content/input/' + f
    shutil.move(f, dst)
    if os.path.splitext(f)[1].lower() in ('.wav','.mp3','.m4a','.ogg','.flac'):
        audio = dst
if audio is None:
    raise ValueError("Nenhum audio enviado. Rode esta celula de novo.")
print('Foto:', photo, '| Audio:', audio)

In [ ]:
# 3) Gera o video: foto + audio (SadTalker)
ok = False
try:
    from gfpgan import GFPGANer
    ok = True
except Exception:
    print('GFPGAN indisponivel - gerando sem realce (qualidade boa)')
%cd /content/SadTalker
if ok:
    !python inference.py --driven_audio "{audio}" --source_image "{photo}" --result_dir /content/output --still --preprocess full --enhancer gfpgan
else:
    !python inference.py --driven_audio "{audio}" --source_image "{photo}" --result_dir /content/output --still --preprocess full
import glob
vids = sorted(glob.glob('/content/output/**/*.mp4', recursive=True))
if vids:
    video = vids[-1]
    print('Video gerado:', video)
else:
    print('Nao gerou - me mande o print do fim da saida acima.')

In [ ]:
# 4) Download do video
from google.colab import files
files.download(video)